# Stride, padding and output shapes

**Learning objective:** Predict convolution output dimensions and verify them with TensorFlow.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:15:37.682315: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974937.697996    3819 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974937.702360    3819 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:15:39.407411: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
configs=[]
for padding in ["valid","same"]:
    for stride in [1,2]:
        layer=tf.keras.layers.Conv2D(8,3,strides=stride,padding=padding)
        y=layer(tf.zeros((1,28,28,1)))
        configs.append({"padding":padding,"stride":stride,"output":str(tuple(y.shape)),"params":layer.count_params()})
display(pd.DataFrame(configs))


,padding,stride,output,params
0,valid,1,"(1, 26, 26, 8)",80
1,valid,2,"(1, 13, 13, 8)",80
2,same,1,"(1, 28, 28, 8)",80
3,same,2,"(1, 14, 14, 8)",80


For `VALID`, spatial size roughly follows $\lfloor (n-k)/s\rfloor+1$. `SAME` chooses padding so stride 1 preserves size. Stride changes spatial sampling; filter count changes channel depth.


In [3]:
n=28;k=3
print("VALID stride 1 formula:",(n-k)//1+1); print("VALID stride 2 formula:",(n-k)//2+1)


VALID stride 1 formula: 26
VALID stride 2 formula: 13
